# `apply` vs `apply_peetre`: Efficiency & Precision Analysis

This notebook compares three ways of applying a pseudo-differential operator:

* **`backend="direct"`** — the generic Kohn-Nirenberg quadrature. This is the **ground truth** used below for error measurement.
* **`backend="peetre"`, `joint_backend="direct"`** — Peetre decomposition for the local/separable part of the symbol (fast FFT path), with the leftover *joint* residual (the part that is genuinely entangled in `x` and `xi` and cannot be separated) applied through the same slow direct quadrature as ground truth.
* **`backend="peetre"`, `joint_backend="lowrank"`** — same Peetre decomposition, but the joint residual is itself approximated by a low-rank sum of separable pairs $\sum_k a_k(x) q_k(\xi)$ (Chebyshev/SVD factorization), so that *every* term — local, separable and joint — is applied via the fast FFT path.

We extend the original symbol set with **joint symbols**: terms such as $1/(1+x^2+\xi^2)$ or $e^{-(x\xi)^2/w}$ that mix space and frequency non-separably (`_peetre_classify_terms` puts these in the `joint` bucket, not `local` or `separable`). For these, `apply_peetre` cannot fully decompose the symbol into fast FFT terms — the interesting question is *how much* is left over, and how well `joint_backend="lowrank"` can approximate it.

### 🐛 Bug Fix Note: The Frequency Grid Misalignment
If you previously ran this with `np.fft.fftshift` applied to `kx` and `ky`, you likely saw massive errors (e.g., `1.46e+04`) for Variable Coefficients, while Constant Coefficients had `0.00e+00` error.

**Why?** The slow path in `apply()` correctly recomputes frequencies internally. But the fast path `_apply_constant_fft()` (used by `apply_peetre()` for the separated $q(\xi)$ multipliers) blindly uses the `kx` array passed to it. Passing a *shifted* `kx` multiplied the unshifted FFT spectrum by the shifted symbol, completely misaligning the frequencies.

**The Fix:** We simply use the **unshifted** `fftfreq` grid below so it perfectly matches the `scipy.fft.fft` convention.

In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
from psiop import PseudoDifferentialOperator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

boundary_condition = 'dirichlet'


## Benchmark harness

`benchmark_configs` runs a list of `apply()` configurations on the same field `u` and returns, for
each configuration, the mean execution time over `repeats` calls and the relative $L^2$ error
against the **first** configuration in the list (`backend="direct"`), which is treated as ground truth.

The three configurations used throughout are:

| label | `backend` | `joint_backend` | meaning |
|---|---|---|---|
| `direct` | `"direct"` | — | ground truth: full direct quadrature |
| `peetre (joint=direct)` | `"peetre"` | `"direct"` | fast FFT for local/separable terms, direct quadrature for the joint residual |
| `peetre (joint=lowrank)` | `"peetre"` | `"lowrank"` | fast FFT for local/separable terms **and** for a low-rank approximation of the joint residual |


In [ ]:
CONFIGS = [
    {'label': 'direct',                 'kwargs': dict(backend='direct')},
    {'label': 'peetre (joint=direct)',  'kwargs': dict(backend='peetre', joint_backend='direct')},
    {'label': 'peetre (joint=lowrank)', 'kwargs': dict(backend='peetre', joint_backend='lowrank')},
]

def make_test_function(op, x_grid, y_grid=None):
    """Same test function u as in the baseline efficiency analysis."""
    if op.dim == 1:
        return np.exp(-x_grid**2) * np.cos(5 * x_grid)
    X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
    return np.exp(-(X**2 + Y**2)) * np.cos(5 * X) * np.cos(5 * Y)


def benchmark_configs(op, u, x_grid, kx, y_grid=None, ky=None, configs=CONFIGS,
                       repeats=3, boundary_condition='periodic'):
    """
    Run each configuration in `configs` on `op.apply(u, ...)`, timing it and
    computing its relative L2 error against the first configuration's output
    (assumed to be backend='direct', i.e. ground truth).

    Returns
    -------
    dict: label -> {'mean_time': float, 'times': list[float], 'rel_l2_error': float}
    """
    results = {}
    ref = None

    for cfg in configs:
        label, kwargs = cfg['label'], cfg['kwargs']

        # Warm-up call (JIT / cache effects, symbolic decomposition caching, etc.)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                     boundary_condition=boundary_condition, **kwargs)

        times = []
        out = None
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for _ in range(repeats):
                t0 = time.perf_counter()
                out = op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky,
                                boundary_condition=boundary_condition, **kwargs, joint_max_rel_error=1e-2)
                times.append(time.perf_counter() - t0)

        mean_t = float(np.mean(times))

        if ref is None:
            ref = out
            err = 0.0
        else:
            norm_ref = np.linalg.norm(ref)
            err = float(np.linalg.norm(out - ref) / norm_ref) if norm_ref > 0 else float(np.linalg.norm(out - ref))

        results[label] = {'mean_time': mean_t, 'times': times, 'rel_l2_error': err}

    return results


## 1D Symbols

We keep the original local/non-local, constant/variable symbols and add three **joint** symbols
that `_peetre_classify_terms` cannot place in `local` or `separable`:

* **Joint (Smooth)**: $e^{-(x\xi)^2/40000}$ — genuinely joint, but smooth and slowly varying relative
  to the $(x,\xi)$ domain spanned by the grid, so a low-rank Chebyshev/SVD factorization should
  approximate it well.
* **Joint (Resolvent)**: $1/(1+x^2+\xi^2)$ — genuinely joint, sharply peaked at $\xi=0$ relative to
  the wide frequency range of the grid ($|\xi|$ up to $\mathcal{O}(N)$), a harder case for low-rank
  factorization at default settings.
* **Joint (Mixed)**: $\xi^2 + 1/(1+x^2+\xi^2)$ — a local (differential) part plus a joint residual,
  so the Peetre decomposition is non-trivial: local part via FFT, residual via direct/low-rank.

In [ ]:
# 1D Symbols
x, xi = sp.symbols('x xi', real=True)

sym_1d_loc_c   = xi**2
sym_1d_loc_v   = (1 + 0.5 * sp.sin(x)) * xi**2
sym_1d_nloc_c  = sp.sqrt(xi**2 + 1.0)
sym_1d_nloc_v  = (1 + 0.5 * sp.sin(x)) * sp.sqrt(xi**2 + 1.0)

# Joint symbols: entangled in x and xi, cannot be separated into a(x) * q(xi)
sym_1d_joint_smooth    = sp.exp(-(x * xi)**2 / 40000)
sym_1d_joint_resolvent = 1 / (1 + x**2 + xi**2)
sym_1d_joint_mixed     = xi**2 + 1 / (1 + x**2 + xi**2)

ops_1d = {
    '1D Local (Const)':      PseudoDifferentialOperator(sym_1d_loc_c, [x], mode='symbol'),
    '1D Local (Var)':        PseudoDifferentialOperator(sym_1d_loc_v, [x], mode='symbol'),
    '1D Non-Local (Const)':  PseudoDifferentialOperator(sym_1d_nloc_c, [x], mode='symbol'),
    '1D Non-Local (Var)':    PseudoDifferentialOperator(sym_1d_nloc_v, [x], mode='symbol'),
    '1D Joint (Smooth)':     PseudoDifferentialOperator(sym_1d_joint_smooth, [x], mode='symbol'),
    '1D Joint (Resolvent)':  PseudoDifferentialOperator(sym_1d_joint_resolvent, [x], mode='symbol'),
    '1D Joint (Mixed)':      PseudoDifferentialOperator(sym_1d_joint_mixed, [x], mode='symbol'),
}


In [ ]:
N_values_1d = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

rows_1d = []

print('Running 1D Benchmarks...')
for N in N_values_1d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)

    # Unshifted frequencies to match scipy.fft.fft output convention
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_1d.items():
        u = make_test_function(op, x_grid)
        res = benchmark_configs(op, u, x_grid, kx, repeats=4, boundary_condition=boundary_condition)

        t_direct = res['direct']['mean_time']
        
        for label, r in res.items():
            speedup = t_direct / r['mean_time'] if r['mean_time'] > 0 else np.nan
            rows_1d.append(dict(
                            N=N, 
                            symbol=name, 
                            backend=label,
                            time=r['mean_time'], 
                            speedup=speedup, 
                            err=r['rel_l2_error']
                        ))
            print(f"N={N:5d} | {name:22s} | {label:24s} | "
                  f"time: {r['mean_time']:.4f}s | err: {r['rel_l2_error']:.2e}")

df_1d = pd.DataFrame(rows_1d)


### 1D Results Table

In [ ]:
pivot_time_1d = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='time')
pivot_err_1d  = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='err')
pivot_speedup_1d = df_1d.pivot_table(index=['symbol', 'N'], columns='backend', values='speedup')

display(pivot_time_1d)
display(pivot_err_1d)
display(pivot_speedup_1d)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

names_1d = list(ops_1d.keys())
cmap = plt.get_cmap('tab10')
colors_1d = {name: cmap(i % 10) for i, name in enumerate(names_1d)}
linestyles = {'direct': '--', 'peetre (joint=direct)': '-', 'peetre (joint=lowrank)': ':'}
markers = {'direct': 'o', 'peetre (joint=direct)': 's', 'peetre (joint=lowrank)': '^'}

for name in names_1d:
    sub = df_1d[df_1d['symbol'] == name]
    for label in ['direct', 'peetre (joint=direct)', 'peetre (joint=lowrank)']:
        s = sub[sub['backend'] == label].sort_values('N')
        ax1.loglog(s['N'], s['time'], marker=markers[label], linestyle=linestyles[label],
                   color=colors_1d[name], label=f'{name} ({label})', alpha=0.85)

ax1.set_xlabel('Grid Size N')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('1D Execution Time: direct vs peetre(joint=direct) vs peetre(joint=lowrank)')
ax1.legend(fontsize=6, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

for name in names_1d:
    sub = df_1d[df_1d['symbol'] == name]
    for label in ['peetre (joint=direct)', 'peetre (joint=lowrank)']:
        s = sub[sub['backend'] == label].sort_values('N')
        ax2.semilogy(s['N'], s['err'].clip(lower=1e-17), marker=markers[label],
                     linestyle=linestyles[label], color=colors_1d[name],
                     label=f'{name} ({label})', alpha=0.85)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error (vs. direct)')
ax2.set_title('1D Precision: peetre backends vs direct ground truth')
ax2.legend(fontsize=6, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

plt.tight_layout()
plt.show()


## 2D Symbols

Same structure as 1D, with genuinely joint 2D symbols added. As in the original notebook, `N` is
kept modest for 2D because `apply(backend="direct")` on spatially-dependent symbols is
$\mathcal{O}(N^4)$ and would be prohibitively slow / memory-hungry beyond a few hundred points.
Because the joint-residual direct path reuses this same $\mathcal{O}(N^4)$ quadrature, the
**`peetre (joint=direct)`** configuration is only as fast as plain `direct` for symbols that are
*fully* joint — the speed-up only appears once `joint_backend="lowrank"` replaces that residual
with fast FFT terms.

In [ ]:
# 2D Symbols
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

sym_2d_loc_c  = xi**2 + eta**2
sym_2d_loc_v  = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)
sym_2d_nloc_c = (xi**2 + eta**2)**0.75
sym_2d_nloc_v = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)**0.75

# Joint symbols
sym_2d_joint_smooth    = sp.exp(-((x * xi)**2 + (y * eta)**2) / 40000)
sym_2d_joint_resolvent = 1 / (1 + x**2 + y**2 + xi**2 + eta**2)

ops_2d = {
    '2D Local (Const)':      PseudoDifferentialOperator(sym_2d_loc_c, [x, y], mode='symbol'),
    '2D Local (Var)':        PseudoDifferentialOperator(sym_2d_loc_v, [x, y], mode='symbol'),
    '2D Non-Local (Const)':  PseudoDifferentialOperator(sym_2d_nloc_c, [x, y], mode='symbol'),
    '2D Non-Local (Var)':    PseudoDifferentialOperator(sym_2d_nloc_v, [x, y], mode='symbol'),
    '2D Joint (Smooth)':     PseudoDifferentialOperator(sym_2d_joint_smooth, [x, y], mode='symbol'),
    '2D Joint (Resolvent)':  PseudoDifferentialOperator(sym_2d_joint_resolvent, [x, y], mode='symbol'),
}


In [ ]:
# We limit N to 128 for 2D because `apply(backend='direct')` on variable / joint symbols
# is O(N^4) and would take too long / OOM for N=256+. With 6 symbols x 3 backends this
# cell can take several minutes -- reduce repeats or N_values_2d if you need a quicker run.
N_values_2d = [16, 32, 64, 128]

rows_2d = []

print('Running 2D Benchmarks... (this may take several minutes, especially for the '
      '`direct` backend on Variable / Joint symbols)')
for N in N_values_2d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    y_grid = -L/2 + dx * np.arange(N)

    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    for name, op in ops_2d.items():
        u = make_test_function(op, x_grid, y_grid)
        res = benchmark_configs(op, u, x_grid, kx, y_grid=y_grid, ky=ky,
                                 repeats=2, boundary_condition=boundary_condition)

        t_direct = res['direct']['mean_time']

        for label, r in res.items():
            speedup = t_direct / r['mean_time'] if r['mean_time'] > 0 else np.nan
            rows_2d.append(dict(
                            N=N, 
                            symbol=name, 
                            backend=label,
                            time=r['mean_time'], 
                            speedup=speedup, 
                            err=r['rel_l2_error']
                        ))
            print(f"N={N:4d}x{N:<4d} | {name:22s} | {label:24s} | "
                  f"time: {r['mean_time']:.4f}s | err: {r['rel_l2_error']:.2e}")

df_2d = pd.DataFrame(rows_2d)


### 2D Results Table

In [ ]:
pivot_time_2d = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='time')
pivot_err_2d  = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='err')
pivot_speedup_2d = df_2d.pivot_table(index=['symbol', 'N'], columns='backend', values='speedup')
display(pivot_time_2d)
display(pivot_err_2d)
display(pivot_speedup_2d)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

names_2d = list(ops_2d.keys())
colors_2d = {name: cmap(i % 10) for i, name in enumerate(names_2d)}

for name in names_2d:
    sub = df_2d[df_2d['symbol'] == name]
    for label in ['direct', 'peetre (joint=direct)', 'peetre (joint=lowrank)']:
        s = sub[sub['backend'] == label].sort_values('N')
        ax1.loglog(s['N'], s['time'], marker=markers[label], linestyle=linestyles[label],
                   color=colors_2d[name], label=f'{name} ({label})', alpha=0.85)

ax1.set_xlabel('Grid Size N (Total Points = N^2)')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('2D Execution Time: direct vs peetre(joint=direct) vs peetre(joint=lowrank)')
ax1.legend(fontsize=6, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

for name in names_2d:
    sub = df_2d[df_2d['symbol'] == name]
    for label in ['peetre (joint=direct)', 'peetre (joint=lowrank)']:
        s = sub[sub['backend'] == label].sort_values('N')
        ax2.semilogy(s['N'], s['err'].clip(lower=1e-17), marker=markers[label],
                     linestyle=linestyles[label], color=colors_2d[name],
                     label=f'{name} ({label})', alpha=0.85)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error (vs. direct)')
ax2.set_title('2D Precision: peetre backends vs direct ground truth')
ax2.legend(fontsize=6, ncol=1, loc='upper left', bbox_to_anchor=(1.01, 1.0))

plt.tight_layout()
plt.show()


## Analysis & Conclusions

### 1. Three regimes, three backends
`_peetre_classify_terms` splits a symbol into **local** (polynomial in $\xi$), **separable**
non-local ($a(x)q(\xi)$), and **joint** (genuinely entangled) terms. `apply(backend="direct")` is
oblivious to this structure and always falls back to the generic quadrature
($\mathcal{O}(N^2)$ in 1D, $\mathcal{O}(N^4)$ in 2D) whenever the symbol depends on space at all —
this is our ground truth throughout. `apply_peetre` instead applies the local/separable part with
the $\mathcal{O}(N\log N)$ FFT path and only needs *some* strategy for whatever joint residual is
left over:

- `joint_backend="direct"` reuses the same slow quadrature for the residual only. For symbols that
  are mostly local/separable (e.g. **1D/2D Joint (Mixed)**, where the residual is a small correction
  to a polynomial principal part) this is still a large win over full `direct`, because only the
  residual — not the whole symbol — pays the quadrature cost. For **fully joint** symbols (e.g.
  **Joint (Smooth)**, **Joint (Resolvent)**), there is no local/separable part to fast-path, so
  `peetre (joint=direct)` costs essentially the same as plain `direct`.
- `joint_backend="lowrank"` additionally factorizes the joint residual into a short sum of separable
  pairs $\sum_k a_k(x) q_k(\xi)$ via a Chebyshev/SVD low-rank approximation, then applies each pair
  through the fast FFT path. This is the only configuration that recovers the full
  $\mathcal{O}(N\log N)$ / $\mathcal{O}(N^2\log N)$ complexity for symbols with a genuinely joint part.

### 2. Precision is not free for the joint residual
Unlike the purely separable case (machine-precision agreement between `apply` and `apply_peetre`),
the low-rank factorization of a joint residual is an **approximation**, and its accuracy depends
strongly on how well-suited the symbol is to a low-degree Chebyshev/SVD fit over the
$(x,\xi)$-box inferred from the grid:

- **Joint (Smooth)**, $e^{-(x\xi)^2/40000}$, varies slowly relative to the $(x,\xi)$ box spanned by
  the grid, so a handful of singular vectors capture it almost exactly — `joint=lowrank` matches
  `direct` to within $10^{-5}$–$10^{-6}$ while running on the fast path.
- **Joint (Resolvent)**, $1/(1+x^2+\xi^2)$, is sharply peaked near $\xi=0$ but the grid's frequency
  range extends to $|\xi|=\mathcal{O}(N)$. A low-degree Chebyshev fit over that whole range struggles
  to resolve the peak, and `joint=lowrank` can show $\mathcal{O}(1)$ relative error at default
  `joint_degree` / `joint_tol`. `joint=direct` remains exact in this regime, at the cost of the slow
  quadrature.

The practical takeaway: `joint_backend="lowrank"` is a genuine complexity win, but — unlike the
separable Peetre path — it trades some accuracy for speed, and that trade-off should be checked
(e.g. via `joint_max_rel_error`, or by tuning `joint_degree` / `joint_tol` / `joint_bounds`) rather
than assumed, especially for symbols that are sharply localized in $\xi$ relative to the resolved
frequency range.

### 3. Practical guidance
- If a symbol is fully local or separable, `apply_peetre` (either `joint_backend`) reproduces
  `apply(backend="direct")` to machine precision at $\mathcal{O}(N\log N)$ cost — no trade-off.
- If a symbol has a small joint residual on top of a dominant local/separable part,
  `joint_backend="direct"` already gives most of the speed-up with zero approximation error in the
  residual term.
- If a symbol is substantially or fully joint, `joint_backend="lowrank"` is required to see any
  speed-up over `direct`, but its error should be checked against the `direct` (or `joint=direct`)
  result, or bounded via `joint_max_rel_error`, before trusting it for production use.